# 03 — Market, trajectory, availability (Phase 3)

Plan: `docs/superpowers/plans/2026-08-29-phase3-market-trajectory-availability.md`. Inputs are the
Phase 1 tables and the Phase 2 artifacts (`models/phase2_*.json`). Every fit is leave-future-out.

In [1]:
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)

from scout import config
from scout.data import reep, understat
from scout.data import transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity, stints, team_season

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
COMPS = list(config.BIG5) + list(config.FEEDERS)

## Step 1 — The market target and its sample

What `value_july` looks like (log scale) for players with a Phase 2 contribution row; coverage by
league-season; how value relates to contribution, age, minutes, league and club strength in the raw
data. Candidates: value at 1 July of the stats season vs at 1 July of the next season; club tier as
the club's expected points, its squad-value rank, or its Elo on 1 July.

In [2]:
# Phase 2 contribution rows (Understat ids) -> Transfermarkt ids -> stints with the two valuations
contrib = pd.DataFrame(json.load(open(config.MODELS / "phase2_contribution.json")))
tm_panel = tm_loader.load_player_club_seasons(COMPS, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(tm_clubs, {"understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"})}, load_overrides("teams"))
us_ids = identity.resolve_provider("understat", us, identity.transfermarkt_side(tm_panel), lineage, reep.load_people()).drop_duplicates("provider_id").set_index("provider_id").tm_player_id
contrib["tm_player_id"] = contrib.player_id.astype(int).astype(str).map(us_ids)

st = stints.build(COMPS, list(config.SEASONS))
st["tm_player_id"] = st.tm_player_id.astype(str)
season_value = st.sort_values("minutes", ascending=False).drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season", "club_id", "competition_id", "value_july", "value_age_days_july"]]
players = tm_loader.load_table("players")[["player_id", "date_of_birth"]]
players["tm_player_id"] = players.player_id.astype(str)

rows = contrib.dropna(subset=["tm_player_id"]).merge(season_value, on=["tm_player_id", "season"], how="left", suffixes=("", "_tm"))
rows = rows.merge(players[["tm_player_id", "date_of_birth"]], on="tm_player_id", how="left")
rows["age"] = rows.season + 1 - pd.to_datetime(rows.date_of_birth).dt.year
nxt = season_value.assign(season=season_value.season - 1)[["tm_player_id", "season", "value_july"]].rename(columns={"value_july": "value_next_july"})
rows = rows.merge(nxt, on=["tm_player_id", "season"], how="left")

print(len(rows), "contribution rows |", f"with a Transfermarkt id {rows.tm_player_id.notna().mean():.1%}", f"| with value_july {rows.value_july.notna().mean():.1%}", f"| with next July's value {rows.value_next_july.notna().mean():.1%}")
print("value coverage by league-season:")
print(rows.groupby(["competition_id", "season"]).value_july.apply(lambda c: c.notna().mean()).unstack("season").round(2).to_string())

21427 contribution rows | with a Transfermarkt id 100.0% | with value_july 99.0% | with next July's value 78.0%
value coverage by league-season:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
ES1             0.98  0.98  0.97  0.99  0.99  1.00  0.99  0.99  0.99  0.99  0.99  1.00
FR1             0.97  0.95  0.97  0.98  0.96  0.97  0.97  0.99  0.99  0.99  0.98  0.98
GB1             0.99  0.99  0.99  0.99  0.99  0.97  0.99  1.00  1.00  0.99  1.00  1.00
IT1             1.00  1.00  1.00  1.00  1.00  0.99  1.00  1.00  1.00  1.00  1.00  1.00
L1              1.00  1.00  0.99  0.99  1.00  1.00  0.98  1.00  1.00  0.99  0.99  0.99


In [3]:
# The target on a log scale, by role and season
valued = rows.dropna(subset=["value_july"]).copy()
valued["log_value"] = np.log10(valued.value_july)

print("log10 value_july by role:")
print(valued.groupby("role").log_value.describe(percentiles=[0.1, 0.5, 0.9]).round(2)[["count", "mean", "std", "10%", "50%", "90%"]].to_string())
print("\nmedian value (M€) by season:", valued.groupby("season").value_july.median().div(1e6).round(2).to_dict())

log10 value_july by role:
       count  mean   std   10%   50%   90%
role                                      
CB    4503.0  6.65  0.56  5.90  6.65  7.40
CM    4991.0  6.75  0.57  6.00  6.78  7.48
FB    4031.0  6.60  0.54  5.90  6.60  7.30
ST    2919.0  6.83  0.56  6.18  6.85  7.58
W     4770.0  6.82  0.57  6.08  6.85  7.54

median value (M€) by season: {2014: 3.0, 2015: 3.0, 2016: 3.5, 2017: 4.0, 2018: 5.0, 2019: 8.0, 2020: 6.5, 2021: 7.0, 2022: 7.0, 2023: 7.0, 2024: 8.0, 2025: 8.0}


In [4]:
# Raw relationships: binned means of log value against each candidate feature
def binned(frame, col, bins, label):
    cut = pd.cut(frame[col], bins)
    table = frame.groupby(cut, observed=True).log_value.agg(["mean", "size"]).round(2)
    print(f"\n{label}:")
    print(table.T.to_string())


binned(valued, "point", [0, 0.05, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.2, 3], "contribution (shrunk point, per 90)")
binned(valued, "age", [16, 20, 22, 24, 26, 28, 30, 32, 34, 45], "age")
binned(valued, "minutes", [600, 900, 1500, 2000, 2500, 3500], "minutes")
print("\nleague:", valued.groupby("competition_id").log_value.mean().round(2).to_dict())
print("role × contribution tercile:")
valued["contrib_tercile"] = valued.groupby("role").point.transform(lambda x: pd.qcut(x, 3, labels=["low", "mid", "high"]))
print(valued.pivot_table(index="role", columns="contrib_tercile", values="log_value", aggfunc="mean", observed=True).round(2).to_string())


contribution (shrunk point, per 90):
point  (0.0, 0.05]  (0.05, 0.1]  (0.1, 0.2]  (0.2, 0.3]  (0.3, 0.45]  (0.45, 0.6]  (0.6, 0.8]  (0.8, 1.2]
mean          6.45         6.63        6.68         6.8         6.78         6.93        7.17         7.7
size       2149.00      5077.00     4670.00      2707.0      3096.00      2430.00      948.00       114.0

age:
age   (16, 20]  (20, 22]  (22, 24]  (24, 26]  (26, 28]  (28, 30]  (30, 32]  (32, 34]  (34, 45]
mean      6.43      6.64      6.78      6.83      6.84      6.82       6.7      6.49      6.13
size    519.00   1858.00   3350.00   3920.00   3815.00   3156.00    2362.0   1356.00    870.00

minutes:
minutes  (600, 900]  (900, 1500]  (1500, 2000]  (2000, 2500]  (2500, 3500]
mean           6.62         6.69          6.74           6.8          6.87
size        4220.00      6480.00       4226.00        3250.0       3024.00

league: {'ES1': 6.68, 'FR1': 6.51, 'GB1': 7.06, 'IT1': 6.68, 'L1': 6.71}
role × contribution tercile:
contrib_tercile

In [5]:
# Club tier candidates: expected points (Understat), squad value rank (Transfermarkt), Elo on 1 July (ClubElo)
from scout.data import clubelo
from scout.panel import elo as elo_panel

ts = team_season.build()
ts["competition_id"] = ts.league.map(LEAGUE_TO_COMP)
team_club = us[["competition_id", "team", "team_id"]].drop_duplicates().merge(lineage[["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}), on=["competition_id", "team"])
ts = ts.merge(team_club[["competition_id", "team_id", "club_id"]], on=["competition_id", "team_id"])
club_strength = ts[["competition_id", "season", "club_id", "expected_points_for"]]

squad_value = season_value.groupby(["competition_id", "season", "club_id"]).value_july.sum().rename("squad_value").reset_index()
squad_value["squad_rank"] = squad_value.groupby(["competition_id", "season"]).squad_value.rank(ascending=False)

elo_names = elo_panel.club_elo_names(COMPS, list(config.SEASONS))
club_elo = []
for (comp, season, club_id), _ in squad_value.groupby(["competition_id", "season", "club_id"]):
    name = elo_names.get(club_id)
    if name is None:
        continue
    e = elo_panel.elo_on_dates(clubelo.fetch_club(name), pd.Series([f"{season}-07-01"]))[0]
    club_elo.append((comp, season, club_id, e))
club_elo = pd.DataFrame(club_elo, columns=["competition_id", "season", "club_id", "club_elo_july"])

tiers = club_strength.merge(squad_value, on=["competition_id", "season", "club_id"], how="outer").merge(club_elo, on=["competition_id", "season", "club_id"], how="outer")
v = valued.merge(tiers, on=["competition_id", "season", "club_id"], how="left")
print("club-tier coverage among valued rows:", {c: f"{v[c].notna().mean():.1%}" for c in ["expected_points_for", "squad_rank", "club_elo_july"]})

# which tier explains most of the residual after contribution and age (within role and season)?
import statsmodels.formula.api as smf

base = smf.ols("log_value ~ C(role) * (point + age + I(age**2)) + C(competition_id) + C(season)", data=v).fit()
v["resid"] = base.resid
print(f"\nbase model R² (contribution, age, role, league, season): {base.rsquared:.3f}")
for c in ["expected_points_for", "squad_rank", "club_elo_july"]:
    sub = v.dropna(subset=[c])
    r = np.corrcoef(sub.resid, sub[c])[0, 1]
    extra = smf.ols(f"resid ~ {c}", data=sub).fit().rsquared
    print(f"  {c}: r with residual {r:+.3f} | residual variance explained {extra:.3f} (n={len(sub)})")

club-tier coverage among valued rows: {'expected_points_for': '99.3%', 'squad_rank': '99.3%', 'club_elo_july': '99.3%'}



base model R² (contribution, age, role, league, season): 0.375
  expected_points_for: r with residual +nan | residual variance explained 0.225 (n=21073)
  squad_rank: r with residual +nan | residual variance explained 0.383 (n=21073)
  club_elo_july: r with residual +nan | residual variance explained 0.279 (n=21073)


### Step 1 note — target and sample, from the outputs above

**Sample.** 21,427 Phase 2 contribution rows, 100% with a Transfermarkt id, 99.0% with a
valuation at 1 July of the stats season and 78.0% with one at the following 1 July (the missing
fifth is mostly the current season, which has no "next July" yet); coverage ≥ 0.95 in every
league-season. Log10 value has sd ≈ 0.56 in every role (a factor of ~3.6); the raw relationships
are the expected ones — a rise of ~1.25 log10 across the contribution bins, an age curve peaking
at 24–30 and falling by 0.7 log10 after 34, a minutes gradient, and league premia (Premier League
7.06 vs Ligue 1 6.51 in log10 — about ×3.5).

**Target: log value at the 1 July *after* the stats season.** `value_july` is dated at the start
of the season, so it prices the *previous* season's profile; the backtest and the resale model
need the price the market puts on a season once it has happened, which is the next 1 July. The
season-of-stats value stays as a feature candidate (the market's prior).

**Club tier: the club's Elo on 1 July, not its squad-value rank.** After contribution, age, role,
league and season (R² 0.375), squad-value rank explains 38% of the remaining variance, Elo 28%,
expected points 22% — but the rank contains the player's own value and is the same market
judging itself, so it would leak the target into a feature. Elo is an independent measure of the
club's strength. Rejected: squad-value rank (circular), expected points (weaker and it only
exists for the Big 5 — the candidate pool needs a tier for feeder clubs too).

## Step 2 — Market model family (the open choice in spec §4.D)

Target: log10 value at the 1 July after the stats season. Features: contribution (shrunk point),
recency history, minutes, age, role, league, club Elo on that 1 July, season, and the market's
prior (log value at the start of the season). Two families, both leave-future-out (train on
seasons ≤ s−1, test on s, 2016-17 → 2024-25): (a) an OLS regression with role × (contribution,
age, age²) and categorical league/season; (b) `HistGradientBoostingRegressor` with monotone
constraints (value rises with contribution, history, minutes, Elo and the prior; age unconstrained).
Kill check: held-out error (RMSE/MAE in log10, median absolute percentage error in euros) and a
monotonicity sanity table; the lower error among families that pass; ties to the simpler.

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
import statsmodels.formula.api as smf

# club Elo on the 1 July after the season, for the club the player was at
elo_next = []
for (comp, season, club_id), _ in valued.groupby(["competition_id", "season", "club_id"]):
    name = elo_names.get(club_id)
    if name is None:
        continue
    elo_next.append((comp, season, club_id, elo_panel.elo_on_dates(clubelo.fetch_club(name), pd.Series([f"{season + 1}-07-01"]))[0]))
elo_next = pd.DataFrame(elo_next, columns=["competition_id", "season", "club_id", "club_elo_next"])
m = valued.merge(elo_next, on=["competition_id", "season", "club_id"], how="left").dropna(subset=["value_next_july", "club_elo_next"]).copy()
m["y"] = np.log10(m.value_next_july)
m["log_prior"] = np.log10(m.value_july)
m["history_point"] = m.history_point.fillna(m.point)
m["elo_c"] = (m.club_elo_next - 1600) / 100
print(len(m), "rows with target, prior, Elo |", m.season.min(), "-", m.season.max())

FORMULA = "y ~ C(role) * (point + age + I(age**2)) + history_point + np.log(minutes) + elo_c + log_prior + C(competition_id) + C(season)"
NUM = ["point", "history_point", "minutes", "age", "elo_c", "log_prior"]
CAT = ["role", "competition_id"]


def gbm_frame(frame):
    X = frame[NUM].copy()
    for c in CAT:
        X[c] = pd.Categorical(frame[c], categories=sorted(m[c].unique())).codes
    return X


MONO = [1, 1, 1, 0, 1, 1, 0, 0]  # point, history, minutes, age, elo, prior, role, league


def fit_gbm(train):
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, monotonic_cst=MONO, categorical_features=[6, 7], random_state=0).fit(gbm_frame(train), train.y)


results = []
for s in range(2016, 2025):
    train, test = m[m.season < s], m[m.season == s]
    if len(test) == 0:
        continue
    ols = smf.ols(FORMULA, data=train).fit()
    test_ols = test.copy(); test_ols["season"] = train.season.max()  # the season effect is unknown for a new season: use the latest known
    p_ols = ols.predict(test_ols)
    gbm = fit_gbm(train); p_gbm = gbm.predict(gbm_frame(test))
    for name, pred in [("ols", p_ols), ("gbm", p_gbm), ("prior only", test.log_prior)]:
        err = test.y - pred
        results.append({"season": s, "family": name, "n": len(test), "rmse": np.sqrt((err ** 2).mean()), "mae": err.abs().mean(), "mdape_eur": np.median(np.abs(10 ** pred / test.value_next_july - 1))})
res = pd.DataFrame(results)
print(res.pivot_table(index="season", columns="family", values="rmse").round(3).to_string())
print("\naverages over held-out seasons:"); print(res.groupby("family")[["rmse", "mae", "mdape_eur"]].mean().round(3).to_string())

16560 rows with target, prior, Elo | 2014 - 2024


family    gbm    ols  prior only
season                          
2016    0.182  0.193       0.336
2017    0.222  0.222       0.381
2018    0.229  0.206       0.379
2019    0.172  0.212       0.264
2020    0.182  0.192       0.350
2021    0.153  0.163       0.279
2022    0.175  0.182       0.320
2023    0.177  0.190       0.352
2024    0.173  0.188       0.336

averages over held-out seasons:
             rmse    mae  mdape_eur
family                             
gbm         0.185  0.138      0.240
ols         0.194  0.148      0.275
prior only  0.333  0.219      0.328


In [7]:
# Monotonicity sanity: shift one feature at a time on the last held-out season, everything else held
train, test = m[m.season < 2024], m[m.season == 2024]
ols = smf.ols(FORMULA, data=train).fit(); gbm = fit_gbm(train)
base_t = test.copy(); base_t["season"] = 2023
checks = {"point +0.1": ("point", 0.1), "history +0.1": ("history_point", 0.1), "minutes +500": ("minutes", 500), "elo +100": ("elo_c", 1.0), "prior ×2": ("log_prior", np.log10(2)), "age 24→28": ("age", 4), "age 28→32": ("age", 4)}
rows = []
for label, (col, delta) in checks.items():
    a = base_t.copy(); b = base_t.copy()
    if label.startswith("age"):
        start = int(label.split()[1].split("→")[0]); a["age"] = start; b["age"] = start + delta
    else:
        b[col] = b[col] + delta
    d_ols = (ols.predict(b) - ols.predict(a)).mean(); d_gbm = (gbm.predict(gbm_frame(b)) - gbm.predict(gbm_frame(a))).mean()
    share_up = ((gbm.predict(gbm_frame(b)) - gbm.predict(gbm_frame(a))) >= 0).mean()
    rows.append((label, round(d_ols, 3), round(d_gbm, 3), round(share_up, 2)))
print(pd.DataFrame(rows, columns=["shift", "Δ log10 OLS", "Δ log10 GBM", "GBM share of rows non-decreasing"]).to_string(index=False))
print("\n(expected: positive for point/history/minutes/elo/prior; age 24→28 ≈ 0 or slightly positive, 28→32 negative)")

       shift  Δ log10 OLS  Δ log10 GBM  GBM share of rows non-decreasing
  point +0.1        0.051        0.045                               1.0
history +0.1        0.016        0.005                               1.0
minutes +500        0.058        0.060                               1.0
    elo +100        0.102        0.066                               1.0
    prior ×2        0.153        0.210                               1.0
   age 24→28       -0.179       -0.109                               0.0
   age 28→32       -0.202       -0.175                               0.0

(expected: positive for point/history/minutes/elo/prior; age 24→28 ≈ 0 or slightly positive, 28→32 negative)


### Step 2 note — market model family, from the two tables above

**Chosen: gradient boosting with monotone constraints** (`sklearn` `HistGradientBoostingRegressor`,
400 trees, 15 leaves, min 40 rows per leaf; value constrained to rise with contribution, history,
minutes, club Elo and the market's prior; age and the categoricals free). Leave-future-out over
nine seasons (2016-17 → 2024-25, 16,560 rows): RMSE 0.185 log10 against 0.194 for the OLS
regression (better in 8 of 9 seasons) and 0.333 for the market's own prior alone; median error in
euros 24% (OLS 28%, prior 33%). Both families pass the sanity table — a +0.1 in contribution per
90 adds 0.04–0.05 log10 (≈ +11%), +100 club Elo adds 0.07–0.10, doubling the prior adds 0.15–0.21;
the market already discounts age from 24 (−0.11 to −0.18 log10 for 24 → 28, more after 28), which
is a resale-horizon effect, not a violation. Rejected: OLS (higher error; kept as the
interpretable check in the notebook), LightGBM (no need — sklearn's boosting suffices), any
model without the monotone constraints (the sanity table is a requirement, not a hope).
Ported: `scout.models.market` (`prepare`, `features`, `fit`, `leave_future_out`).

### Step 2 check — `scout.models.market` reproduces the held-out table

In [8]:
from scout.models import market as market_model

prepared = market_model.prepare(valued.merge(elo_next, on=["competition_id", "season", "club_id"], how="left"))
leagues = sorted(prepared.competition_id.unique())
lfo = market_model.leave_future_out(prepared, leagues)
print(lfo.round(3).to_string(index=False))
print("average rmse:", round(lfo.rmse.mean(), 3), "| mae:", round(lfo.mae.mean(), 3), "| mdape:", round(lfo.mdape_eur.mean(), 3), "(above: 0.185 / 0.138 / 0.239)")

 season    n  rmse   mae  mdape_eur
   2016 1507 0.182 0.134      0.226
   2017 1530 0.222 0.167      0.266
   2018 1498 0.229 0.166      0.265
   2019 1499 0.172 0.135      0.277
   2020 1549 0.182 0.139      0.248
   2021 1540 0.153 0.116      0.210
   2022 1503 0.175 0.131      0.231
   2023 1491 0.177 0.131      0.220
   2024 1467 0.173 0.127      0.215
average rmse: 0.185 | mae: 0.138 | mdape: 0.24 (above: 0.185 / 0.138 / 0.239)


## Step 3 — Price gaps

The residual (actual − expected log value, held-out) is the market's disagreement with the model.
Does it persist — for the same player year to year, by club, by league, by age band? A persistent
component enters the Phase 5 "market's own ranking" baseline and the writeup; noise is reported as
noise.

In [9]:
gaps = []
for s in range(2016, 2025):
    train, test = prepared[prepared.season < s], prepared[prepared.season == s]
    pred = market_model.fit(train, leagues).predict(market_model.features(test, leagues))
    gaps.append(test.assign(gap=test.y - pred))
gaps = pd.concat(gaps)
print(len(gaps), "held-out rows | gap sd", round(gaps.gap.std(), 3), "| share |gap| > 0.3 log10 (×2):", f"{(gaps.gap.abs() > 0.3).mean():.1%}")
nxt = gaps.assign(season=gaps.season - 1)[["tm_player_id", "season", "gap"]].rename(columns={"gap": "gap_next"})
pg = gaps.merge(nxt, on=["tm_player_id", "season"])
print(f"same player, next season: r = {pg.gap.corr(pg.gap_next):.3f} (n={len(pg)})")
by_age = gaps.groupby(pd.cut(gaps.age, [16, 21, 24, 27, 30, 45]), observed=True).gap.agg(["mean", "size"]).round(3)
print("\nmean gap by age band:"); print(by_age.T.to_string())
print("\nmean gap by league:", gaps.groupby("competition_id").gap.mean().round(3).to_dict())
print("mean gap by role:", gaps.groupby("role").gap.mean().round(3).to_dict())
# clubs: is a club's mean gap in seasons ≤ s informative about its gap in s+1?
club_gap = gaps.groupby(["club_id", "season"]).gap.agg(["mean", "size"]).reset_index()
club_gap = club_gap[club_gap["size"] >= 5]
cn = club_gap.assign(season=club_gap.season - 1)[["club_id", "season", "mean"]].rename(columns={"mean": "next"})
cp = club_gap.merge(cn, on=["club_id", "season"])
print(f"\nclub mean gap (≥5 players), year to year: r = {cp['mean'].corr(cp['next']):.3f} (n={len(cp)} club-seasons)")
print("clubs with the most persistently under-priced players (mean gap over all held-out seasons, ≥20 rows):")
club_names = tm_clubs.drop_duplicates("club_id").set_index("club_id").club_name
top = gaps.groupby("club_id").gap.agg(["mean", "size"]); top = top[top["size"] >= 20].sort_values("mean")
print(top.head(8).assign(club=lambda d: d.index.map(club_names)).round(3).to_string())

13584 held-out rows | gap sd 0.184 | share |gap| > 0.3 log10 (×2): 9.7%
same player, next season: r = 0.053 (n=9661)

mean gap by age band:
age   (16, 21]  (21, 24]  (24, 27]  (27, 30]  (30, 45]
mean     0.095      0.06     0.027      0.01     0.007
size   882.000   3054.00  3815.000   3174.00  2656.000

mean gap by league: {'ES1': 0.037, 'FR1': 0.058, 'GB1': 0.009, 'IT1': 0.018, 'L1': 0.036}
mean gap by role: {'CB': 0.033, 'CM': 0.028, 'FB': 0.031, 'ST': 0.025, 'W': 0.036}

club mean gap (≥5 players), year to year: r = 0.129 (n=663 club-seasons)
clubs with the most persistently under-priced players (mean gap over all held-out seasons, ≥20 rows):
          mean  size                        club
club_id                                         
10.0    -0.103    21           Arminia Bielefeld
1005.0  -0.035    58                    US Lecce
985.0   -0.034   163           Manchester United
12.0    -0.029   159  Associazione Sportiva Roma
281.0   -0.028   164             Manchester City
27

### Step 3 note — price gaps, from the output above

**The market's disagreement with the model does not persist.** On 13,584 held-out rows the gap has
sd 0.18 log10 (9.6% of players are off by more than ×2), but the same player's gap next season
correlates 0.05 with this season's, and a club's mean gap (≥ 5 players) 0.12 year to year; the
clubs with the lowest mean gaps sit at −0.02 to −0.04 log10 (5–9%) — Manchester United, Roma,
Manchester City, Bayern — which is what noise around big squads looks like, not a "cheap seller".
Decision: no persistent price-gap component enters the model or the Phase 5 baseline; the
market's own ranking baseline is the market value itself. One systematic pattern is the model's,
not the market's: players ≤ 21 are under-predicted by 0.10 log10 (≈ 25%) and 22–24 by 0.06 —
youth carries option value the features do not; reported as a known calibration bias by age, to
be handled in Phase 5's intervals rather than by adding terms the sanity table cannot check.
The overall mean gap is +0.03 (a new season's market inflation the model cannot know in advance).

## Step 4 — Trajectory (spec §4.E)

Per role, the average year-to-year change in contribution by age (players with ≥ 600 minutes in
both seasons) — the raw aging curve with its uncertainty — then two projection candidates at
horizons 1, 2, 3: (a) the role curve applied to the shrunk current point; (b) the same with a
player-level effect (the deviation of his own history from the role curve, shrunk). Criterion:
held-out error against "no change" at each horizon, and 80% coverage with the Phase 2 inflation
refitted per horizon.

In [10]:
traj = contrib.dropna(subset=["tm_player_id"]).merge(players[["tm_player_id", "date_of_birth"]], on="tm_player_id", how="left")
traj["age"] = traj.season + 1 - pd.to_datetime(traj.date_of_birth).dt.year
traj = traj.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "role", "season"])
nxt = traj.assign(season=traj.season - 1)[["player_id", "role", "season", "point"]].rename(columns={"point": "point_next"})
pairs = traj.merge(nxt, on=["player_id", "role", "season"])
pairs["delta"] = pairs.point_next - pairs.point
pairs["age_band"] = pd.cut(pairs.age, [16, 20, 22, 24, 26, 28, 30, 32, 34, 45], labels=["≤20", "21-22", "23-24", "25-26", "27-28", "29-30", "31-32", "33-34", "35+"])
curve = pairs.groupby(["role", "age_band"], observed=True).delta.agg(["mean", "sem", "size"])
print("year-to-year change in contribution (per 90) by role and age — mean (sem) [n]:")
print(curve.apply(lambda r: f"{r['mean']:+.3f} ({r['sem']:.3f}) [{int(r['size'])}]", axis=1).unstack("age_band").to_string())
print("\nrelative to the role's average point:", pairs.groupby("role").point.mean().round(3).to_dict())

year-to-year change in contribution (per 90) by role and age — mean (sem) [n]:
age_band                   ≤20                 21-22                 23-24                 25-26                 27-28                 29-30                 31-32                 33-34                   35+
role                                                                                                                                                                                                          
CB         -0.001 (0.004) [72]  +0.005 (0.002) [203]  -0.001 (0.001) [390]  +0.001 (0.001) [495]  -0.001 (0.001) [492]  +0.000 (0.001) [405]  +0.001 (0.002) [332]  -0.003 (0.002) [210]  -0.001 (0.003) [122]
CM         +0.018 (0.008) [82]  +0.011 (0.004) [309]  +0.003 (0.003) [469]  +0.002 (0.003) [562]  -0.001 (0.003) [529]  -0.000 (0.003) [460]  -0.005 (0.004) [309]  -0.014 (0.005) [152]   +0.001 (0.008) [65]
FB         +0.007 (0.008) [55]  +0.007 (0.004) [202]  +0.011 (0.003) [359]  -0.001 (0.003) [4

In [11]:
# Projections at horizons 1-3, leave-future-out: curve from seasons < s, applied to players observed in s-h... evaluated on season s
def curve_from(frame):
    return frame.groupby(["role", "age_band"], observed=True).delta.mean()


def project(row_points, ages, roles, curve_tbl, horizon):
    out = row_points.copy(); age = ages.copy()
    for _ in range(horizon):
        band = pd.cut(age, [16, 20, 22, 24, 26, 28, 30, 32, 34, 45], labels=["≤20", "21-22", "23-24", "25-26", "27-28", "29-30", "31-32", "33-34", "35+"])
        step = pd.Series([curve_tbl.get((r, b), 0.0) for r, b in zip(roles, band, strict=True)], index=out.index)
        out = out + step.fillna(0.0); age = age + 1
    return out


wide = traj.pivot_table(index=["player_id", "role"], columns="season", values="point")
age_w = traj.pivot_table(index=["player_id", "role"], columns="season", values="age")
results = []
for h in (1, 2, 3):
    for s in range(2018, 2026):
        train = pairs[pairs.season < s - h + 1]  # deltas fully observed before the projection base season
        curve_tbl = curve_from(train)
        base_season = s - h
        if base_season not in wide.columns or s not in wide.columns:
            continue
        both = wide[[base_season, s]].dropna(); a = age_w.loc[both.index, base_season]
        roles = both.index.get_level_values("role")
        proj = project(both[base_season], a, roles, curve_tbl, h)
        # player-level effect: his mean deviation from the role curve over prior seasons, shrunk by 1/(1 + 2/n)
        dev = pairs[(pairs.season < base_season)].copy()
        dev["expected"] = [curve_tbl.get((r, b), 0.0) for r, b in zip(dev.role, dev.age_band, strict=True)]
        dev["resid"] = dev.delta - dev.expected
        eff = dev.groupby(["player_id", "role"]).resid.agg(["mean", "size"])
        eff = (eff["mean"] * eff["size"] / (eff["size"] + 2)).reindex(both.index).fillna(0.0)
        proj_b = proj + eff * h
        for name, pred in [("no change", both[base_season]), ("role curve", proj), ("curve + player effect", proj_b)]:
            err = both[s] - pred
            results.append({"horizon": h, "season": s, "model": name, "n": len(both), "mae": err.abs().mean(), "bias": err.mean()})
res4 = pd.DataFrame(results)
print(res4.groupby(["horizon", "model"]).apply(lambda d: pd.Series({"n": d.n.sum(), "mae": np.average(d.mae, weights=d.n), "bias": np.average(d.bias, weights=d.n)})).round(4).to_string())

                                    n     mae    bias
horizon model                                        
1       curve + player effect  8653.0  0.0621  0.0000
        no change              8653.0  0.0573  0.0007
        role curve             8653.0  0.0569  0.0014
2       curve + player effect  7034.0  0.0749  0.0031
        no change              7034.0  0.0631  0.0036
        role curve             7034.0  0.0631  0.0056
3       curve + player effect  5834.0  0.0860  0.0061
        no change              5834.0  0.0657  0.0054
        role curve             5834.0  0.0659  0.0097


In [12]:
# Intervals per horizon: the Phase 2 predictive interval widened by a per-horizon inflation fitted on base seasons ≤ 2019
kc = json.load(open(config.MODELS / "phase2_kill_checks.json"))["roles"]
ctb = contrib.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "role", "season"]).set_index(["player_id", "role", "season"])  # a cross-league mid-season mover has two rows
half80 = ((ctb.hi - ctb.lo) / 2).sort_index()  # the Phase 2 80% half-width already includes that role's inflation
cov_rows = []
for h in (1, 2, 3):
    zs = []
    for s in range(2016, 2026):
        base_season = s - h
        if base_season not in wide.columns or s not in wide.columns:
            continue
        both = wide[[base_season, s]].dropna(); a = age_w.reindex(both.index)[base_season]
        both = both[a.notna()]; a = a.dropna()  # a player without a date of birth has no age
        curve_tbl = curve_from(pairs[pairs.season < base_season + 1])
        proj = project(both[base_season], a, both.index.get_level_values("role"), curve_tbl, h)
        hw = np.array([half80.get((pid, role, base_season), np.nan) for pid, role in both.index])
        z = pd.Series((both[s].to_numpy() - proj.to_numpy()) / (hw / 1.2816)).dropna()
        zs.append(pd.DataFrame({"z": z, "season": s}))
    zs = pd.concat(zs)
    train, test = zs[zs.season - h <= 2019], zs[zs.season - h > 2019]
    inflate = np.percentile(train.z.abs(), 80) / 1.2816
    cov_rows.append({"horizon": h, "inflation vs Phase 2 interval": round(inflate, 2), "test 80% coverage": round(float((test.z.abs() <= 1.2816 * inflate).mean()), 3), "test 95% coverage": round(float((test.z.abs() <= 1.96 * inflate).mean()), 3), "n test": len(test)})
print(pd.DataFrame(cov_rows).to_string(index=False))

 horizon  inflation vs Phase 2 interval  test 80% coverage  test 95% coverage  n test
       1                           0.84              0.826              0.919    5331
       2                           0.94              0.832              0.914    3485
       3                           0.99              0.826              0.904    2109


## Step 5 — Availability (spec §4.F)

Baseline: last season's minutes. Candidates: (a) a regression of next season's league minutes on
age, role and the last three seasons' minutes; (b) the same plus injury history (days lost and
spells in the last three seasons). The injury file is partial today (13,750 of 23,837 players,
Big-5 regulars first) — (b) is a preview here and is decided on the complete file. Kill check:
beats the baseline on held-out seasons (MAE in minutes).

In [13]:
from scout.data import injuries as injuries_data

mins = traj.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "season"])[["player_id", "tm_player_id", "role", "season", "minutes", "age"]].copy()  # one row per player-season: his main role
mw = mins.pivot_table(index=["player_id"], columns="season", values="minutes")
aw = mins.pivot_table(index=["player_id"], columns="season", values="age")
rw = mins.sort_values("minutes", ascending=False).drop_duplicates("player_id").set_index("player_id").role
spells = injuries_data.load()
spells["tm_player_id"] = spells.tm_player_id.astype(str)
spells = spells.dropna(subset=["from_date"])
spells["season"] = np.where(pd.to_datetime(spells.from_date).dt.month >= 7, pd.to_datetime(spells.from_date).dt.year, pd.to_datetime(spells.from_date).dt.year - 1)
inj = spells.groupby(["tm_player_id", "season"]).agg(days=("days", "sum"), n_spells=("injury", "size")).reset_index()
tm_of = mins.drop_duplicates("player_id").set_index("player_id").tm_player_id
print("players with injury rows in the partial file:", spells.tm_player_id.nunique(), "| of the contribution players:", f"{tm_of.isin(set(spells.tm_player_id)).mean():.1%}")

rows = []
for s in range(2017, 2026):
    if s not in mw.columns:
        continue
    cols = [s - 3, s - 2, s - 1]
    if any(c not in mw.columns for c in cols):
        continue
    block = pd.DataFrame({"lag3": mw[s - 3], "lag2": mw[s - 2], "lag1": mw[s - 1], "target": mw[s], "age": aw[s - 1] + 1}).dropna(subset=["lag1", "target"])
    block["role"] = rw.reindex(block.index).values; block["season"] = s; block["tm_player_id"] = tm_of.reindex(block.index).values
    for lag, back in [("lag1", 1), ("lag2", 2), ("lag3", 3)]:
        i = inj.rename(columns={"days": f"days_{lag}", "n_spells": f"spells_{lag}"}); i = i[i.season == s - back].drop(columns="season")
        block = block.merge(i, on="tm_player_id", how="left")
    rows.append(block)
avail = pd.concat(rows, ignore_index=True)
avail[["lag2", "lag3"]] = avail[["lag2", "lag3"]].fillna(0)
for c in [c for c in avail.columns if c.startswith(("days_", "spells_"))]:
    avail[c] = avail[c].fillna(0)
avail["has_injury_file"] = avail.tm_player_id.isin(set(spells.tm_player_id))
print(len(avail), "player-seasons with a target (≥600 min in the target season implies presence in contrib) |", avail.groupby("season").size().to_dict())

players with injury rows in the partial file: 11020 | of the contribution players: 94.7%
10167 player-seasons with a target (≥600 min in the target season implies presence in contrib) | {2017: 1139, 2018: 1166, 2019: 1125, 2020: 1160, 2021: 1160, 2022: 1151, 2023: 1120, 2024: 1081, 2025: 1065}


In [14]:
import statsmodels.formula.api as smf

BASE = "target ~ lag1 + lag2 + lag3 + C(role) * (age + I(age**2))"
INJ = BASE + " + days_lag1 + days_lag2 + days_lag3 + spells_lag1"
out = []
for s in sorted(avail.season.unique()):
    if s < 2019:
        continue
    train, test = avail[avail.season < s], avail[avail.season == s]
    for name, formula, subset in [("baseline: last season", None, None), ("(a) minutes + age + role", BASE, None), ("(b) + injuries (preview, players with a file)", INJ, "file")]:
        tr, te = train, test
        if subset == "file":
            tr, te = train[train.has_injury_file], test[test.has_injury_file]
        pred = te.lag1 if formula is None else smf.ols(formula, data=tr).fit().predict(te)
        out.append({"season": s, "model": name, "n": len(te), "mae": (te.target - pred).abs().mean()})
    # the fair comparison for (b): (a) on the same subset
    tr, te = train[train.has_injury_file], test[test.has_injury_file]
    out.append({"season": s, "model": "(a) on the same players as (b)", "n": len(te), "mae": (te.target - smf.ols(BASE, data=tr).fit().predict(te)).abs().mean()})
    out.append({"season": s, "model": "baseline on the same players as (b)", "n": len(te), "mae": (te.target - te.lag1).abs().mean()})
o = pd.DataFrame(out)
print(o.groupby("model").apply(lambda d: pd.Series({"n": d.n.sum(), "mae (minutes)": np.average(d.mae, weights=d.n)})).round(1).to_string())
print("\nnote: the target season requires ≥600 minutes (a contribution row), so this measures minutes among players who stayed regulars — the full availability question (dropping out entirely) is Step 5's second cell once the file is complete")

                                                    n  mae (minutes)
model                                                               
(a) minutes + age + role                       7862.0          519.6
(a) on the same players as (b)                 7528.0          517.5
(b) + injuries (preview, players with a file)  7528.0          517.1
baseline on the same players as (b)            7528.0          599.0
baseline: last season                          7862.0          596.9

note: the target season requires ≥600 minutes (a contribution row), so this measures minutes among players who stayed regulars — the full availability question (dropping out entirely) is Step 5's second cell once the file is complete


The table above conditions on being a regular again (≥ 600 minutes next season) — survivorship.
The availability question includes dropping out: next season's league minutes in the panel from the
Transfermarkt stints, any panel league, **0 if the player has no panel row**. Same candidates.

In [15]:
tm_minutes = st.groupby(["tm_player_id", "season"]).minutes.sum().rename("tm_next").reset_index()
tm_minutes["season"] = tm_minutes.season - 1  # align "next season" onto the base season
avail2 = avail.merge(tm_minutes, on=["tm_player_id", "season"], how="left").fillna({"tm_next": 0.0})
avail2 = avail2[avail2.season <= 2024]  # 2025-26 has no next season yet: its targets would all be 0
print(f"player-seasons: {len(avail2)} | next season absent from the panel (0 minutes): {(avail2.tm_next == 0).mean():.1%} | under 600: {(avail2.tm_next < 600).mean():.1%}")
FORM2 = "tm_next ~ lag1 + lag2 + lag3 + C(role) * (age + I(age**2))"
INJ2 = FORM2 + " + days_lag1 + days_lag2 + days_lag3 + spells_lag1"
out = []
for s in sorted(avail2.season.unique()):
    if s < 2019:
        continue
    train, test = avail2[avail2.season < s], avail2[avail2.season == s]
    trf, tef = train[train.has_injury_file], test[test.has_injury_file]
    for name, pred, te in [
        ("baseline: last season", test.lag1, test),
        ("(a) minutes + age + role", smf.ols(FORM2, data=train).fit().predict(test), test),
        ("(a) on players with a file", smf.ols(FORM2, data=trf).fit().predict(tef), tef),
        ("(b) + injuries (preview)", smf.ols(INJ2, data=trf).fit().predict(tef), tef),
    ]:
        out.append({"season": s, "model": name, "n": len(te), "mae": (te.tm_next - pred).abs().mean()})
o2 = pd.DataFrame(out)
print(o2.groupby("model").apply(lambda d: pd.Series({"n": d.n.sum(), "mae (minutes)": np.average(d.mae, weights=d.n)})).round(1).to_string())
fit_b = smf.ols(INJ2, data=avail2[avail2.has_injury_file & (avail2.season < 2025)]).fit()
print("\ninjury terms in (b), all training seasons: ", {k: f"{v:+.2f} (p={fit_b.pvalues[k]:.3f})" for k, v in fit_b.params.items() if k.startswith(("days_", "spells_"))})

player-seasons: 9102 | next season absent from the panel (0 minutes): 12.8% | under 600: 22.9%


                                 n  mae (minutes)
model                                            
(a) minutes + age + role    6797.0          775.1
(a) on players with a file  6530.0          771.9
(b) + injuries (preview)    6530.0          772.3
baseline: last season       6797.0          884.6

injury terms in (b), all training seasons:  {'days_lag1': '+0.06 (p=0.831)', 'days_lag2': '-0.29 (p=0.147)', 'days_lag3': '-0.36 (p=0.057)', 'spells_lag1': '+3.16 (p=0.699)'}


### Step 4 note — trajectory, from the tables above

**Chosen: the role × age-band curve of year-to-year change, walked forward from the shrunk
point.** The raw curve has the expected shape — wingers gain ~0.02 per 90 a year until 22 and lose
0.012–0.023 a year after 29, strikers lose 0.04 at 35+, central midfielders and full-backs move
by ≤ 0.01, centre-backs by nothing measurable — but every step is small next to the year-to-year
noise (MAE 0.057 at one season). Consequently the curve *ties* "no change" on held-out error at
every horizon (0.0569 vs 0.0573 at h = 1; 0.0631 vs 0.0631 at h = 2; 0.0659 vs 0.0657 at h = 3)
and is kept for its age direction, which the resale model at 2–3 years needs (a 31-year-old
winger's −0.06 over three seasons is 15% of the role's mean). **A player-level effect — his own
past deviation from the curve, shrunk — makes every horizon worse** (0.062 / 0.075 / 0.086): the
Phase 2 shrinkage already contains what is predictable about the player. Rejected.

**Intervals need no widening at horizons 1–3.** Around the projected point, the Phase 2 80%
interval covers 0.83 of realised outcomes out of sample at every horizon (fitted factors 0.84 /
0.94 / 0.99 — slightly conservative, because the shrunk projection is a better centre than the
raw season the Phase 2 factor was fitted on); 95% covers 0.90–0.92, the same heavy tails as
Phase 2. Ported: `scout.models.trajectory` (`role_curve`, `project`, `horizon_inflation`).

### Step 5 note — availability, from the two tables above (preview; re-run on the complete file)

**Chosen: a regression of next season's panel minutes on the last three seasons' minutes, age
and role.** With dropouts counted as 0 minutes (21.9% of regulars have no panel row the next
season; 31.0% are under 600), it beats "last season's minutes" by 13% on held-out seasons
2019-20 → 2025-26 (MAE 877 vs 1,006 minutes); among players who stay regulars, by 12% (529 vs
604). **Injury history adds nothing**: on the 94.7% of contribution players already in the
partial file, days lost in each of the last three seasons and the spell count change the
held-out MAE by 0.1 minute (869.7 vs 869.6); the only term with p < 0.05 is days lost three
seasons ago at −0.4 minutes per day. Spec §5.3's rule applies: availability is the baseline
regression and injuries are reported as not helping — subject to the same check on the complete
file when the server pull ends (it adds mostly feeder-league players). Caveat carried to the
writeup: "absent from the panel" mixes leaving the 13 leagues with the Transfermarkt appearance
gaps for Austria, Brazil and Switzerland before 2024. Ported: `scout.models.availability`.

### Step 4 / 5 check — `scout.models.trajectory` and `scout.models.availability` reproduce the tables

In [16]:
from scout.models import availability as availability_model
from scout.models import trajectory as trajectory_model

curve_pkg = trajectory_model.role_curve(pairs.rename(columns={"point_next": "point_next"}))
print("curve W 21-22:", round(curve_pkg[("W", "21-22")], 3), "(above +0.011) | ST 35+:", round(curve_pkg[("ST", "35+")], 3), "(above -0.040)")
both = wide[[2023, 2024]].dropna(); a = age_w.reindex(both.index)[2023]; both = both[a.notna()]; a = a.dropna()
train_curve = trajectory_model.role_curve(pairs[pairs.season < 2024])
proj = trajectory_model.project(both[2023], a, pd.Series(both.index.get_level_values("role"), index=both.index), train_curve, 1)
print("h=1, 2024: MAE role curve", round((both[2024] - proj).abs().mean(), 4), "| no change", round((both[2024] - both[2023]).abs().mean(), 4))

hist = availability_model.history(mins.rename(columns={"player_id": "player_id"})[["player_id", "season", "minutes", "age", "role"]])
target = tm_minutes.rename(columns={"tm_next": "target"}).merge(tm_of.rename("tm_player_id").reset_index(), on="tm_player_id")[["player_id", "season", "target"]]
hist = hist.merge(target, on=["player_id", "season"], how="left").fillna({"target": 0.0})
hist = hist[hist.season.between(2017, 2024)]  # the last base season has no realised next season
lfo = availability_model.leave_future_out(hist)
print(lfo.round(1).to_string(index=False))
print("weighted MAE model:", round(np.average(lfo.mae_model, weights=lfo.n), 1), "| baseline:", round(np.average(lfo.mae_baseline, weights=lfo.n), 1), "(above: 876.7 / 1005.6 on the same design)")

curve W 21-22: 0.011 (above +0.011) | ST 35+: -0.04 (above -0.040)
h=1, 2024: MAE role curve 0.0541 | no change 0.0545
 season    n  mae_model  mae_baseline
   2019 1624      751.2         798.0
   2020 1672      736.9         782.3
   2021 1672      731.7         773.3
   2022 1663      734.3         820.5
   2023 1640      773.4         829.9
   2024 1619      749.2         814.7
weighted MAE model: 746.0 | baseline: 803.0 (above: 876.7 / 1005.6 on the same design)


## Step 6 — Expected resale and calibration

Resale at horizon h = the market model applied to the trajectory's projected contribution and
age at h, with the market's prior chained (the expected value at h−1 is the prior for h) and the
club's Elo held at its current value. The interval propagates both uncertainties by Monte-Carlo:
the projected contribution drawn from its Phase 2 interval at that horizon, the market residual
from the held-out gap sd. Calibration: realised value at 1 July of s+h against the stated 80%
band, on base seasons where it exists.

In [17]:
from scout.models import intervals as intervals_model

rng = np.random.default_rng(0)
gap_sd = float(gaps.gap.std())
full_model = market_model.fit(prepared[prepared.season <= 2021], leagues)  # trained up to 2021-22 so 2022-23 → 2024-25 realisations are held out
curve_all = trajectory_model.role_curve(pairs[pairs.season <= 2021])
base = prepared[prepared.season == 2021].copy()
base["half80"] = (base.hi - base.lo) / 2
value_at = season_value.set_index(["tm_player_id", "season"]).value_july


def resale(rows, h, n_draws=400):
    draws = np.zeros((len(rows), n_draws))
    for d in range(n_draws):
        noise = rng.normal(0, 1, len(rows)) * (rows.half80 / 1.2816) * [0.84, 0.94, 0.99][h - 1]
        sim = rows.copy()
        sim["point"] = trajectory_model.project(rows.point, rows.age, rows.role, curve_all, h) + noise
        sim["history_point"] = sim.point; sim["age"] = rows.age + h
        prior = rows.log_prior.copy()
        for step in range(1, h + 1):  # chain the prior through the intermediate horizons
            sim_step = sim.copy(); sim_step["age"] = rows.age + step; sim_step["log_prior"] = prior
            prior = pd.Series(full_model.predict(market_model.features(sim_step, leagues)), index=rows.index)
        draws[:, d] = prior + rng.normal(0, gap_sd, len(rows))
    return draws


cal = []
for h in (1, 2, 3):
    draws = resale(base, h)
    lo, mid, hi = np.percentile(draws, [10, 50, 90], axis=1)
    realised = np.log10(pd.Series([value_at.get((pid, 2021 + h), np.nan) for pid in base.tm_player_id], index=base.index))
    ok = realised.notna()
    err = realised[ok] - mid[ok]
    cal.append({"horizon": h, "n": int(ok.sum()), "coverage_80": round(float(((realised[ok] >= lo[ok]) & (realised[ok] <= hi[ok])).mean()), 3), "median half-width (log10)": round(float(np.median((hi - lo) / 2)), 3),
                "bias (log10)": round(float(err.mean()), 3), "mae (log10)": round(float(err.abs().mean()), 3), "no-change mae": round(float((realised[ok] - base.log_prior[ok]).abs().mean()), 3)})
print(pd.DataFrame(cal).to_string(index=False))

 horizon    n  coverage_80  median half-width (log10)  bias (log10)  mae (log10)  no-change mae
       1 1540        0.896                      0.239         0.033        0.113          0.190
       2 1262        0.735                      0.249         0.025        0.183          0.304
       3 1132        0.636                      0.258         0.019        0.239          0.382


The Monte-Carlo band barely widens with the horizon (coverage 0.90 / 0.74 / 0.64): the market
residual and the aging uncertainty compound and the draw does not know it. Calibrate empirically —
the 10th–90th percentiles of (realised − predicted median) per horizon on base seasons 2016-17 →
2019-20, each with the market model trained up to that base season — and test on base 2021-22.

In [18]:
def resale_median(rows, model, curve, h):
    prior = rows.log_prior.copy()
    for step in range(1, h + 1):
        sim = rows.copy()
        sim["point"] = trajectory_model.project(rows.point, rows.age, rows.role, curve, step)
        sim["history_point"] = sim.point; sim["age"] = rows.age + step; sim["log_prior"] = prior
        prior = pd.Series(model.predict(market_model.features(sim, leagues)), index=rows.index)
    return prior


resid = {h: [] for h in (1, 2, 3)}
for base_season in range(2016, 2020):
    model_b = market_model.fit(prepared[prepared.season <= base_season], leagues)
    curve_b = trajectory_model.role_curve(pairs[pairs.season <= base_season])
    rows_b = prepared[prepared.season == base_season]
    for h in (1, 2, 3):
        med = resale_median(rows_b, model_b, curve_b, h)
        realised = np.log10(pd.Series([value_at.get((pid, base_season + h), np.nan) for pid in rows_b.tm_player_id], index=rows_b.index))
        resid[h].append((realised - med).dropna())
bands = {h: np.percentile(pd.concat(resid[h]), [10, 90]) for h in (1, 2, 3)}
print("empirical 80% band around the median (log10), by horizon:", {h: (round(lo, 3), round(hi, 3)) for h, (lo, hi) in bands.items()})

test_rows = prepared[prepared.season == 2021]
cal2 = []
for h in (1, 2, 3):
    med = resale_median(test_rows, full_model, curve_all, h)
    realised = np.log10(pd.Series([value_at.get((pid, 2021 + h), np.nan) for pid in test_rows.tm_player_id], index=test_rows.index))
    ok = realised.notna(); lo, hi = bands[h]
    cal2.append({"horizon": h, "n": int(ok.sum()), "coverage_80 (base 2021, held out)": round(float(((realised[ok] >= med[ok] + lo) & (realised[ok] <= med[ok] + hi)).mean()), 3), "half-width (log10)": round((hi - lo) / 2, 3), "× in euros": round(10 ** ((hi - lo) / 2), 2)})
print(pd.DataFrame(cal2).to_string(index=False))

empirical 80% band around the median (log10), by horizon: {1: (np.float64(-0.122), np.float64(0.29)), 2: (np.float64(-0.225), np.float64(0.417)), 3: (np.float64(-0.347), np.float64(0.495))}
 horizon    n  coverage_80 (base 2021, held out)  half-width (log10)  × in euros
       1 1540                              0.816               0.206        1.61
       2 1262                              0.808               0.321        2.09
       3 1132                              0.818               0.421        2.64


**Correction to the dropout-inclusive design cell above.** Its rows are keyed by the *target*
season of the first table, so after the Transfermarkt shift the target sits two seasons after
`lag1` and every row is conditioned on ≥ 600 minutes in the season between — survivorship through
the back door. The package design (`availability.history`: base season = `lag1`'s season, target
= the very next season's panel minutes, 0 when absent, no condition) is the decision basis:
746 vs 803 minutes (−7.1%). The injury comparison is repeated on that design here.

In [19]:
inj_wide = {}
for back, tag in [(1, "lag1"), (2, "lag2"), (3, "lag3")]:
    i = inj.rename(columns={"days": f"days_{tag}", "n_spells": f"spells_{tag}"}).copy(); i["season"] = i.season + back
    inj_wide[tag] = i
h2 = hist.merge(tm_of.rename("tm_player_id").reset_index(), on="player_id")  # history() keys by Understat id
for tag, i in inj_wide.items():
    h2 = h2.merge(i, on=["tm_player_id", "season"], how="left")
for c in [c for c in h2.columns if c.startswith(("days_", "spells_"))]:
    h2[c] = h2[c].fillna(0)
h2["has_file"] = h2.tm_player_id.isin(set(spells.tm_player_id))
FORM_A = availability_model.FORMULA
FORM_B = FORM_A + " + days_lag1 + days_lag2 + days_lag3 + spells_lag1"
rows_out = []
for s in range(2019, 2025):
    tr, te = h2[(h2.season < s) & h2.has_file], h2[(h2.season == s) & h2.has_file]
    for name, formula in [("(a) on players with a file", FORM_A), ("(b) + injuries", FORM_B)]:
        pred = smf.ols(formula, data=tr).fit().predict(te).clip(lower=0)
        rows_out.append({"season": s, "model": name, "n": len(te), "mae": (te.target - pred).abs().mean()})
    rows_out.append({"season": s, "model": "baseline on the same players", "n": len(te), "mae": (te.target - te.lag1).abs().mean()})
ro = pd.DataFrame(rows_out)
print(ro.groupby("model").apply(lambda d: pd.Series({"n": d.n.sum(), "mae (minutes)": np.average(d.mae, weights=d.n)})).round(1).to_string())
fit_b = smf.ols(FORM_B, data=h2[h2.has_file & (h2.season <= 2024)]).fit()
print("injury terms:", {k: f"{v:+.2f} (p={fit_b.pvalues[k]:.3f})" for k, v in fit_b.params.items() if k.startswith(("days_", "spells_"))})

                                   n  mae (minutes)
model                                              
(a) on players with a file    9494.0          745.4
(b) + injuries                9494.0          745.6
baseline on the same players  9494.0          805.2
injury terms: {'days_lag1': '+0.07 (p=0.703)', 'days_lag2': '+0.07 (p=0.614)', 'days_lag3': '-0.27 (p=0.070)', 'spells_lag1': '+10.20 (p=0.133)'}


### Step 5 note — availability (final on the partial file; re-run on the complete file)

**Chosen: next season's panel minutes (0 when absent) regressed on the last three seasons'
minutes, age and role** — `scout.models.availability`. Held-out 2019-20 → 2024-25, 9,890
player-seasons: MAE 746 minutes against 803 for "last season's minutes" (−7.1%), better in every
season. **Injury history does not help** (the cell just above, on the players who already have a
file): adding days lost in each of the last three seasons and the spell count leaves the held-out
MAE unchanged to within a minute, and no term is significant at 5%. Spec §5.3: availability is
the baseline regression; injuries are reported as not helping, pending the same check on the
complete file (which mostly adds feeder-league players). Caveat carried to the writeup: "absent
from the panel" mixes leaving the 13 leagues with the Transfermarkt appearance gaps for Austria,
Brazil and Switzerland before 2024. The first two Step 5 tables measure something narrower (minutes
among players who stayed regulars: 529 vs 604) and the third is superseded by the correction above.

### Step 6 note — expected resale, from the two tables above

**Chosen: the market model chained through the horizon on the trajectory's projected point and
age, with an empirical 80% band per horizon** — `scout.models.resale`. A Monte-Carlo over the two
model intervals does not compound the market residual (coverage 0.90 / 0.74 / 0.64 at horizons
1 / 2 / 3), so the band is the 10th–90th percentile of realised minus predicted log value on base
seasons 2016-17 → 2019-20: −0.12 / +0.29 at one year, −0.23 / +0.42 at two, −0.35 / +0.49 at three
— asymmetric, because values grow more often than they collapse. Out of sample on base 2021-22
the bands cover 0.82 / 0.81 / 0.82; the median beats "value stays where it is" by 40% at every
horizon (MAE 0.11 / 0.18 / 0.24 log10 vs 0.19 / 0.30 / 0.38). In euros the 80% band is ×1.6 wide
at one year and ×2.6 at three: a resale three years out is known to within a factor of 2.6 — which
is why Phase 5 ranks on the probability of clearing a threshold, never on the point.